# CFNA: corpus → parameters (Colab training journal)

Turns the open-source corpus stack into trained CFNA weights, then runs the
conversational SFT stage and lets you chat with the result — all resumable, so
a Colab disconnect never loses more than one save interval.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine).

Honest framing (see `PLAN.md` / `docs/BREAKTHROUGH_MAP.md`): SFT *shapes*
competence but cannot *create* it. The gate is **base held-out bpb < 1.8**
before SFT; the 35M-rung acceptance bars are listed at the bottom.

In [ ]:
# 1) Setup: clone the repo and install deps (Colab already ships CUDA torch)
import os
REPO = "LMMinier/nueronce"
BRANCH = "claude/cfna-neural-core-verify-ldvgn3"  # or your default branch
if not os.path.exists("nueronce"):
    !git clone --branch $BRANCH https://github.com/$REPO nueronce
%cd nueronce
!git pull
%pip install -q numpy datasets pytest cryptography cffi
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2) Safety gate: the fp16/AMP numerics tests must be green before any AMP run
!python -m pytest tests/test_gpu_amp.py tests/test_prompting.py tests/test_chat_format_drift.py -q

In [ ]:
# 3) (Recommended) Mount Google Drive so checkpoints survive disconnects
USE_DRIVE = True
CKPT_ROOT = "checkpoints"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_ROOT = "/content/drive/MyDrive/cfna_checkpoints"
    os.makedirs(CKPT_ROOT, exist_ok=True)
print("checkpoints ->", CKPT_ROOT)

In [ ]:
# 4) Download the corpus stack (multi-subject open-source pull, ~400 MB)
#    Sources/licenses: cfna/corpus/stack.py (22 entries — stories, encyclopedic,
#    math, code, science QA, dialogue). Every 20th doc goes to validation.
!python scripts/dump_corpus_stack.py --out corpus_stack --phase 2 \
    --target-bytes 400000000 --val-every 20
!du -sh corpus_stack && ls corpus_stack/stack_text | head

In [ ]:
# 5) (Optional) Commit the downloaded corpus to GitHub on a dedicated branch.
#    Files >90 MB are split (GitHub hard-limits 100 MB per file). Uses a
#    fine-grained token with repo write access — prompted, never stored.
PUSH_CORPUS = True
if PUSH_CORPUS:
    from getpass import getpass
    token = getpass("GitHub token (repo write): ")
    !find corpus_stack -name '*.txt' -size +90M -exec sh -c 'split -b 90m "$1" "$1.part-" && rm "$1"' _ {} \;
    !git config user.email "luisminier79@gmail.com" && git config user.name "LMMinier"
    !git checkout -B corpus-stack
    !git add -f corpus_stack && git commit -m "Add downloaded corpus stack (split >90MB files)" || echo 'nothing to commit'
    !git push -u https://$token@github.com/$REPO corpus-stack
    !git checkout $BRANCH
    del token
# To restore split files later: cat file.txt.part-* > file.txt

In [ ]:
# 6) BASE PRETRAIN (the long stage): corpus bytes -> parameters.
#    base_35m on a T4 with AMP; re-run this cell after any disconnect —
#    --resume continues from the same checkpoint. Repeat until held-out
#    bpb plateaus (target <= 1.5; gate for SFT is < 1.8).
BASE_CKPT = f"{CKPT_ROOT}/cfna_base_35m.pt"
!python scripts/train_checkpoint.py --preset base_35m --corpus corpus_stack \
    --minutes 170 --seq 192 --batch 16 --lr 3e-4 --amp --resume --out "$BASE_CKPT"

In [ ]:
# 7) Build the mixed conversational SFT dataset (54K+ records, canonical
#    format, <=25% per register, leakage-checked). Deterministic; ~10 s.
!python scripts/build_conversation_sft.py --out-dir data/conversation_sft

In [ ]:
# 8) CONVERSATIONAL SFT: response-masked loss on the canonical format,
#    warm-started from the base checkpoint. best.pt is best-by-val.
#    Only run once step 6's held-out bpb is < 1.8 — SFT cannot rescue a weak base.
SFT_DIR = f"{CKPT_ROOT}/conv_35m"
!python scripts/train_conversation.py --data data/conversation_sft \
    --preset base_35m --init-from "$BASE_CKPT" --loss response \
    --out-dir "$SFT_DIR" --minutes 60 --batch 16 --lr 1e-4 --amp
# resume instead of warm-start on later runs: swap --init-from for --resume

In [ ]:
# 9) Evaluate: inference suite + teacher-forced byte accuracy came from
#    training logs; this runs the phase-2 behavioural suite on best.pt.
!python scripts/eval_inference_phase2.py --write-suite --checkpoint "$SFT_DIR/best.pt" || true

In [ ]:
# 10) CHAT with the model (canonical format, stamped in checkpoint meta)
from cfna.chat import Conversation, load_checkpoint
model, ckpt = load_checkpoint(f"{SFT_DIR}/best.pt")
print("prompt_format:", ckpt.get("meta", {}).get("prompt_format"),
      "| val acc:", ckpt.get("meta", {}).get("best_val_acc"))
convo = Conversation(model, system="You are CFNA.", temperature=0.0)
for q in ["Hello", "What is two plus two?", "What is the capital of France?",
          "Is 8 even or odd?", "Thank you"]:
    print(f"User: {q}\nCFNA: {convo.say(q)}\n")

In [ ]:
# 11) Push training metrics + curves back to the repo (never the raw weights;
#     those live in Drive). The cloud session reads these for go/no-go.
from getpass import getpass
token = getpass("GitHub token (repo write): ")
!git add metrics/ && git commit -m "Add Colab training metrics" || echo 'nothing to commit'
!git push https://$token@github.com/$REPO $BRANCH
del token

In [ ]:
# 12) (Optional) Push the trained checkpoint itself to GitHub on a
#     dedicated branch, split into <90 MB chunks (GitHub hard-limits 100 MB
#     per file). Drive stays the primary store; this is the shareable copy.
PUSH_WEIGHTS = True
if PUSH_WEIGHTS:
    from getpass import getpass
    token = getpass("GitHub token (repo write): ")
    !mkdir -p trained_weights
    !cp "$SFT_DIR/best.pt" trained_weights/conv_35m_best.pt
    !cp "$BASE_CKPT" trained_weights/cfna_base_35m.pt
    !find trained_weights -name '*.pt' -size +90M -exec sh -c 'split -b 90m "$1" "$1.part-" && rm "$1"' _ {} \;
    !git checkout -B trained-weights
    !git add -f trained_weights && git commit -m "Add trained checkpoints (split >90MB files)" || echo 'nothing to commit'
    !git push -u https://$token@github.com/$REPO trained-weights
    !git checkout $BRANCH
    del token
# Restore later on any machine:  cat conv_35m_best.pt.part-* > conv_35m_best.pt

## Gates and acceptance (from `PLAN.md` / `docs/CODEX_HANDOFF.md`)

1. **Do not SFT until base held-out bpb < 1.8** — keep re-running cell 6.
2. 35M-rung acceptance: bpb ≤ 1.5 · choice-ranking beats chance by ≥15 pts on
   ≥3 MCQ subjects · ≥60% valid non-echo stop-terminated answers · 5/5
   grammatical transcripts. Only then move to `--preset base_90m`.
3. Never eval a checkpoint in a prompt format it wasn't trained on
   (`docs/FORMAT.md`).
4. Report weak results in `docs/reports/` — they are part of the record.